# Aufgabe 2 – Hodgkin-Huxley-Gleichungssystem

Euler & RK4 (selbst), Vergleich mit scipy.odeint, Stromvariation, Stromimpuls.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt


## 2a) Euler-Verfahren, 50 ms, konstanter Strom $I=I_0 

In [ ]:
from src import hodgkin_huxley as hh

t = np.arange(0, 50, 0.01)
f = lambda y, t: hh.rhs(y, t, I_ext=-5.0)
y = hh.solve_euler(f, hh.initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, Euler, I₀ = −5 nA (Ruhezustand)")

## 2b) RK4 – Implementierung

In [ ]:


t = np.arange(0, 50, 0.01)
f = lambda y, t: hh.rhs(y, t, I_ext=-5.0)
y = hh.solve_rk4(f, hh.initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, RK4, I₀ = −5 nA (Ruhezustand)")

## 2b) Stabilitätsvergleich - Euler vs. RK4

In [ ]:

I0 = 10.0                          # spikeauslösender Strom -> steile Flanken fordern den Solver
dts = [0.01, 0.05, 0.08, 0.09]

fig, (axE, axR) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for dt in dts:
    t = np.arange(0, 50, dt)
    f = lambda y, t: hh.rhs(y, t, I0)
    UE = hh.solve_euler(f, hh.initial_state(), t)[:, 0]
    UR = hh.solve_rk4(f,   hh.initial_state(), t)[:, 0]
    axE.plot(t, UE, label=f"dt={dt}")
    axR.plot(t, UR, label=f"dt={dt}")

for ax, titel in ((axE, "Euler"), (axR, "RK4")):
    ax.set_title(titel); ax.set_xlabel("t [ms]"); ax.legend()
    ax.set_ylim(-100, 120)         # begrenzen, sonst zerdrückt die explodierende Kurve alles
axE.set_ylabel("U [mV]")
plt.tight_layout()
plt.show()

## 2b) odeint - Implementierung

In [ ]:
from scipy.integrate import odeint

t = np.arange(0, 50, 0.01)
f = lambda y, t: hh.rhs(y, t, I_ext=-5.0)
y = odeint(f, hh.initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, odeint, I₀ = −5 nA (Ruhezustand)")

## 2b) Performance Vergleich

In [ ]:
import time

def zeit(fn, wiederholungen=10):
    t0 = time.perf_counter()
    for _ in range(wiederholungen):
        fn()
    return (time.perf_counter() - t0) / wiederholungen * 1000  # ms pro Durchlauf

verfahren = {
    "Euler":  lambda: hh.solve_euler(f, hh.initial_state(), t),
    "RK4":    lambda: hh.solve_rk4(f, hh.initial_state(), t),
    "odeint": lambda: odeint(hh.rhs, hh.initial_state(), t, args=(-5.0,)),
}

for name, fn in verfahren.items():
    print(f"{name:8s}: {zeit(fn):.3f} ms")

## 2c) Variation von $I_0$ zwischen -5 nA und 15 nA mit RK4

In [ ]:

I_0 = [-5.0, 0.0, 2.2, 5.0, 10.0, 15.0]  # Stromstärken in nA
t = np.arange(0, 50, 0.01)

fig, axes = plt.subplots(len(I_0), 1, figsize=(8, 1.8*len(I_0)), sharex=True)
for ax, I in zip(axes, I_0):
    f = lambda y, t: hh.rhs(y, t, I)          # eigener Solver (RK4)
    U = hh.solve_rk4(f, hh.initial_state(), t)[:, 0]
    ax.plot(t, U)
    ax.set_ylabel("U [mV]")
    ax.set_title(f"I₀ = {I} nA", loc="left", fontsize=10)
    ax.axhline(0, color="gray", lw=0.5, ls="--")   # Orientierung: Spike-Schwelle grob bei 0 mV
axes[-1].set_xlabel("t [ms]")
plt.tight_layout()
plt.show()


**Erklärung**

Bei Variation des konstanten Stroms $I_0$ zeigt sich ein klares Schwellenverhalten.
Für kleine Ströme ($I_0 \lesssim 2$ nA, insbesondere der Grundstrom $I_0 = -5$ nA)
bleibt das Neuron inaktiv: Die Membranspannung verharrt nahe dem Ruhepotential bzw.
zeigt nur eine kleine unterschwellige Auslenkung, aber kein Aktionspotential.
Der Reiz reicht nicht aus, um die Natriumkanäle ausreichend zu öffnen.

Oberhalb einer Schwelle von etwa $I_0 \approx 2.3$ nA wird ein Aktionspotential
ausgelöst: $U$ steigt sprunghaft auf $\approx +38$ mV an und fällt anschließend unter
das Ruhepotential zurück (Nachhyperpolarisation), bevor es sich wieder erholt. Für
noch größere Ströme feuert das Neuron periodisch, und die Feuerrate steigt mit
$I_0$** (in 50 ms z. B. 1 Spike bei 3 nA, 2 bei 6 nA, 4 bei 10 nA). Dies entspricht der
klassischen Frequenz-Strom-Beziehung: Ein stärkerer Dauerreiz führt nicht zu höheren
Spitzen, sondern zu häufigeren Aktionspotentialen. In den folgenden Aufgaben wird
wieder fest $I_0 = -5$ nA verwendet.

## 2d) Stromimpuls (t=10..11 ms, I_imp=50 nA): U, I, n, m, h

In [ ]:
def stromimpuls(t):
    if 10 <= t <= 11:
        return 50.0
    else:
        return -5.0

t = np.arange(0, 50, 0.01)
f = lambda y, t: hh.rhs(y, t, stromimpuls(t))
y = hh.solve_rk4(f, hh.initial_state(), t)
U = y[:, 0]
I = np.array([stromimpuls(ti) for ti in t])
n = y[:, 1]  # Na⁺-Kanäle
m = y[:, 2]  # K⁺-Kanäle
h = y[:, 3]  # Inaktivierung Na⁺-Kanäle

groessen = [(U, "U [mV]"), (I, "I [nA]"), (n, "n"), (m, "m"), (h, "h")]

fig, axes = plt.subplots(5, 1, figsize=(8, 9), sharex=True)
for ax, (daten, label) in zip(axes, groessen):
    ax.plot(t, daten)
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("t [ms]")
axes[0].set_title("2d) Stromimpuls: I₀ = −5 nA, Impuls 50 nA bei t = 10–11 ms", loc="left")
plt.tight_layout()
plt.show()

**2d) Stromimpuls und Funktion der Gatingvariablen**

Ausgehend vom Ruhezustand ($I_0 = -5$ nA) wird für $t \in [10, 11]$ ms ein kurzer,
starker Impuls von $I_\mathrm{imp} = 50$ nA angelegt. Dieser depolarisiert die Membran
so weit, dass ein einzelnes Aktionspotential ausgelöst wird. Der zeitliche Ablauf
lässt sich vollständig über die drei Gatingvariablen verstehen, die auf sehr
unterschiedlichen Zeitskalen reagieren:

- **$m$ ($\mathrm{Na}^+$-Aktivierung)** hat die schnellste Zeitkonstante. Bei der
  Depolarisation öffnet $m$ fast augenblicklich, die Natriumkanäle leiten, und der
  einströmende $\mathrm{Na}^+$-Strom treibt $U$ steil nach oben (Aufstrich des Spikes
  bis $\approx +40$ mV) — ein selbstverstärkender Prozess.
- **$h$ ($\mathrm{Na}^+$-Inaktivierung)** reagiert langsamer und fällt während des
  Spikes ab. Dadurch werden die Natriumkanäle wieder geschlossen und der
  $\mathrm{Na}^+$-Einstrom gestoppt.
- **$n$ ($\mathrm{K}^+$-Aktivierung)** steigt ebenfalls verzögert an, öffnet die
  Kaliumkanäle, und der ausströmende $\mathrm{K}^+$-Strom repolarisiert die Membran —
  $U$ fällt wieder ab.

Das Zusammenspiel „schnelles $m$ gegen langsames $h$ und $n$" erklärt die Form des
Aktionspotentials: schneller Anstieg durch $\mathrm{Na}^+$, anschließende Repolarisation
durch $\mathrm{K}^+$. Da $n$ nach dem Spike noch erhöht und $h$ noch niedrig ist,
unterschreitet $U$ kurzzeitig das Ruhepotential. 